# P&L Attribution Tear Sheet — Inline Rendering

A **P&L attribution tear sheet** decomposes an instrument's profit-and-loss between two
dates (T₀ → T₁) into the factors that drove it: **carry** (coupon income, roll-down,
theta), **rates**, **credit**, **FX**, **vol**, and a residual. The headline is a
**contribution waterfall** that bridges from zero through each factor to the total P&L,
backed by a factor table and a carry / roll-down breakdown.

`reporting.attribution_tearsheet(attribution, ...)` is a pure formatter over a
`PnlAttribution` (produced by `attribution.attribute_pnl`). Like the other sheets, the
returned `TearSheet` renders inline in Jupyter via `_repr_html_` — just make it the last
expression in a cell. You can also pass `instrument=` plus two markets to have it compute
the attribution for you.

In [ ]:
import datetime as dt
import json
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.attribution import PnlAttribution, attribute_pnl
from finstack_quant import reporting

In [ ]:
# A USD fixed-rate bond: ACME 4.25% maturing 2034-03-15.
bond = json.dumps({"type": "bond", "spec": {
    "id": "ACME 4.25% 2034",
    "notional": {"amount": "10000000", "currency": "USD"},
    "issue_date": "2024-03-15",
    "maturity": "2034-03-15",
    "cashflow_spec": {"Fixed": {
        "coupon_type": "Cash", "rate": 0.0425,
        "freq": {"count": 6, "unit": "months"},
        "dc": "Thirty360", "bdc": "following",
        "calendar_id": "weekends_only", "stub": "None",
        "end_of_month": False, "payment_lag_days": 0}},
    "discount_curve_id": "USD-OIS",
    "call_put": None,
    "attributes": {"tags": [], "meta": {}},
    "pricing_overrides": {}}})


# Two market snapshots one month apart on the same USD-OIS curve.
# Subtracting from the discount factors at T1 lifts implied rates (a modest sell-off).
def usd_ois(as_of: str, shift: float = 0.0) -> str:
    mc = MarketContext()
    mc.insert(DiscountCurve(
        "USD-OIS", date.fromisoformat(as_of),
        [(0.0, 1.0), (0.5, 0.98 - shift), (1.0, 0.96 - shift), (2.0, 0.92 - shift),
         (3.0, 0.88 - shift), (5.0, 0.80 - shift), (10.0, 0.65 - shift)],
        day_count="act_365f"))
    return mc.to_json()


t0, t1 = "2025-01-15", "2025-02-18"
market_t0 = usd_ois(t0)
market_t1 = usd_ois(t1, shift=0.004)
print(f"Attribution window: {t0} → {t1}")

In [ ]:
# Decompose the P&L between the two dates with a parallel (full-reprice) attribution.
attribution = PnlAttribution.from_json(
    attribute_pnl(bond, market_t0, market_t1, t0, t1, "Parallel"))

cur = attribution.currency
print(f"Total P&L:  {attribution.total_pnl:>12,.2f} {cur}")
print(f"  Carry:    {attribution.carry:>12,.2f}")
print(f"  Rates:    {attribution.rates_curves_pnl:>12,.2f}")
print(f"  Residual: {attribution.residual:>12,.2f}")
print(f"Repricings: {attribution.num_repricings}")

In [ ]:
# Full attribution tear sheet — rendered inline via _repr_html_.
# The contribution waterfall bridges from 0 through each factor to Total P&L; below it,
# a factor table and the carry / roll-down breakdown (the deferred instrument-sheet detail).
reporting.attribution_tearsheet(attribution, generated=dt.date(2026, 6, 21))